In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

# Get project root by navigating up from this notebook's location
# Notebook is in src/pipeline/, so go up 2 levels to reach project root
current_dir = Path(os.getcwd())
if 'notebooks' in str(current_dir) or 'pipeline' in str(current_dir):
    # Try to find project root by looking for src directory
    project_root = current_dir
    while project_root != project_root.parent:
        if (project_root / 'src').exists() and (project_root / 'data').exists():
            break
        project_root = project_root.parent
else:
    # If we're already at project root or somewhere else, search for it
    project_root = current_dir
    while project_root != project_root.parent:
        if (project_root / 'src').exists() and (project_root / 'data').exists():
            break
        project_root = project_root.parent

project_root_str = str(project_root.resolve())

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

# Change to project root directory for relative paths
os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Update projected starting lineups

In [3]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 10 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'data/raw/player_lines/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'data/raw/player_lines/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Joel Embiid,Over,18.5,-137,2025-12-05,2025-12-05T00:08:14Z
1,Underdog,player_points,Joel Embiid,Under,18.5,-137,2025-12-05,2025-12-05T00:08:14Z
2,Underdog,player_points,Paul George,Over,15.5,-137,2025-12-05,2025-12-05T00:08:14Z
3,Underdog,player_points,Paul George,Under,15.5,-137,2025-12-05,2025-12-05T00:08:14Z
4,Underdog,player_points,Draymond Green,Over,9.5,-137,2025-12-05,2025-12-05T00:08:14Z


In [5]:
from src.features.feature_engine import FeatureEngine

# Initialize FeatureEngine with NGBOOST model for points prediction
engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "ngboost_model_paths": {
        "mean_model": "src/models/saved/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl",
        "variance_model": "src/models/saved/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl",
        "calibration_factor": "src/models/saved/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl",
        "calibration_params": "src/models/saved/NGBOOST_PTS_CALIBRATION_PARAMS_PRODUCTION.pkl",  # Add this
        "features": "src/models/saved/pts_features.pkl"
    }
})

# Test prediction for a single player
result = engine.project_player(
    player_name="LaMelo Ball",
    data=s26,
    date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print("Prediction Result:")
print(f"  Predicted Minutes: {result['predicted_minutes']:.2f}")
print(f"  Predicted Usage: {result['predicted_usage']:.3f}")
print(f"  Predicted Points: {result['predicted_points']:.2f}")
print(f"\nFull result: {result}")

Prediction Result:
  Predicted Minutes: 28.61
  Predicted Usage: 0.286
  Predicted Points: 23.44

Full result: {'predicted_minutes': 28.61181640625, 'predicted_usage': 0.28637197613716125, 'predicted_points': 23.44025925624402}


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 61 players...
Processing 58 players...
Generated 1510 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
1236,Ben Saraf,Derik Queen,6.5,14.5,-137,-137,17.39,24.57,0.967,0.929,over,over,1,164.38,0.822,Med,High
1079,Terance Mann,Jaden McDaniels,9.5,13.5,-137,-137,18.49,21.71,0.934,0.925,over,over,1,154.13,0.771,Med,Med
887,Sam Hauser,Donte DiVincenzo,10.5,13.5,-137,-137,19.19,21.25,0.917,0.903,over,over,1,143.44,0.717,Med,Med
1280,Day'Ron Sharpe,Mike Conley,7.5,4.5,-137,-137,15.30,8.61,0.901,0.891,over,over,1,136.04,0.680,Med,Low
1124,Tyrese Martin,Gabe Vincent,11.5,6.5,-137,-137,20.14,12.42,0.891,0.862,over,over,1,126.03,0.630,High,Med


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24)]

prizepicksPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 67 players...
Processing 63 players...
Generated 1782 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
1522,Day'Ron Sharpe,Jaden McDaniels,6.5,13.5,-137,-115,14.18,21.78,over,over,0.902,0.914,0.8085,0.338,0.392,0.513,142.54,0.713,1,5.68,5.70,Med,Med,"(3.0, 25.3)","(10.6, 32.9)",0.05,0,142.5
725,Payton Pritchard,Donte DiVincenzo,22.5,13.5,-112,-110,16.91,21.12,under,over,0.870,0.885,0.7551,0.355,0.374,0.491,126.54,0.633,1,5.40,5.93,Med,Med,"(6.3, 27.5)","(9.5, 32.7)",0.05,0,126.5
1383,Tyrese Martin,Derik Queen,10.0,14.5,-137,-120,18.18,23.16,over,over,0.870,0.871,0.7424,0.306,0.339,0.441,122.72,0.614,1,6.93,7.23,High,High,"(4.6, 31.8)","(9.0, 37.3)",0.05,0,122.7
1475,Svi Mykhailiuk,Mike Conley,8.5,4.5,110,-105,13.83,8.37,over,over,0.825,0.865,0.6999,0.361,0.366,0.467,109.98,0.550,0,5.23,3.09,Med,Low,"(3.6, 24.1)","(2.3, 14.4)",0.05,0,110.0
1494,Isaiah Collier,Saddiq Bey,7.0,14.5,-137,-115,11.61,23.07,over,over,0.822,0.864,0.6962,0.258,0.343,0.401,108.86,0.544,1,4.54,7.35,Low,High,"(2.7, 20.5)","(8.7, 37.5)",0.05,0,108.9
1341,Ziaire Williams,Deandre Ayton,10.5,15.5,106,-113,15.73,11.63,over,under,0.807,0.810,0.6406,0.333,0.292,0.394,92.17,0.461,0,5.50,4.94,Med,Low,"(5.0, 26.5)","(1.9, 21.3)",0.05,0,92.2
332,VJ Edgecombe,Jordan Hawkins,10.5,7.0,-110,-137,17.48,11.66,over,over,0.800,0.800,0.6276,0.289,0.236,0.338,88.29,0.441,1,8.11,5.13,High,Med,"(1.6, 33.4)","(1.6, 21.7)",0.05,0,88.3
508,Quinten Post,Brice Sensabaugh,7.5,9.5,-115,-110,11.88,13.94,over,over,0.772,0.768,0.5808,0.250,0.257,0.313,74.25,0.371,1,5.49,5.49,Med,Med,"(1.1, 22.6)","(3.2, 24.7)",0.05,0,74.2
430,Draymond Green,Naz Reid,9.5,13.5,-110,-110,13.57,17.43,over,over,0.724,0.724,0.5136,0.213,0.213,0.251,54.08,0.270,0,6.38,5.80,High,Med,"(1.1, 26.1)","(6.1, 28.8)",0.05,0,54.1
130,Quentin Grimes,Josh Minott,16.5,7.5,-123,-115,13.20,10.25,under,over,0.711,0.715,0.4987,0.173,0.193,0.217,49.60,0.248,0,6.59,4.04,High,Low,"(0.3, 26.1)","(2.3, 18.2)",0.05,0,49.6


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 61 players...
Processing 58 players...
Generated 23238 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
8577,Andre Drummond,Ben Saraf,Jaden McDaniels,5.5,6.5,13.5,11.68,15.61,21.78,0.890,0.933,0.914,over,over,over,1,309.96,0.620,Low,Med,Med
14467,Payton Pritchard,Day'Ron Sharpe,Donte DiVincenzo,22.5,7.5,13.5,16.91,14.18,21.12,0.870,0.867,0.885,under,over,over,1,260.73,0.521,Med,Med,Med
22505,Svi Mykhailiuk,Deandre Ayton,Derik Queen,8.5,15.5,14.5,13.83,11.63,23.16,0.825,0.810,0.871,over,under,over,0,214.40,0.429,Med,Low,High
4374,VJ Edgecombe,Tyrese Martin,Mike Conley,10.5,11.5,4.5,17.48,18.18,8.37,0.800,0.817,0.865,over,over,over,0,205.70,0.411,High,High,Low
2141,Paul George,Ziaire Williams,Saddiq Bey,15.5,10.5,14.5,22.27,15.73,23.07,0.793,0.807,0.864,over,over,over,1,198.63,0.397,High,Med,High


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 66 players...
Processing 62 players...
Generated 28362 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
16099,Payton Pritchard,Day'Ron Sharpe,Jaden McDaniels,22.5,6.5,13.5,16.91,14.18,21.78,0.870,0.902,0.914,under,over,over,1,287.74,0.575,Med,Med,Med
25918,Tyrese Martin,Deandre Ayton,Donte DiVincenzo,10.0,15.5,13.5,18.18,11.63,21.12,0.870,0.810,0.885,over,under,over,0,236.83,0.474,High,Low,Med
7661,VJ Edgecombe,Svi Mykhailiuk,Derik Queen,10.5,8.5,14.5,17.48,13.83,23.16,0.800,0.825,0.871,over,over,over,1,210.71,0.421,High,Med,High
12076,Quinten Post,Isaiah Collier,Saddiq Bey,7.5,7.0,14.5,11.88,11.61,23.07,0.772,0.822,0.864,over,over,over,1,196.09,0.392,Med,Low,High
9755,Draymond Green,Ziaire Williams,Jordan Hawkins,9.5,10.5,7.0,13.57,15.73,11.66,0.724,0.807,0.800,over,over,over,1,152.42,0.305,High,Med,Med
